# 10장 · 상관관계와 회귀 — 머신러닝의 시작 📈

웹 챕터 **"상관관계와 회귀"** 와 짝을 이루는 Python 실습이자, 3부의 **마지막 노트북**입니다.
상관계수를 구하고, 최소제곱 회귀 직선을 손으로·`np.polyfit` 으로·**경사하강법(2부!)** 으로
세 번 구해서 전부 같은 답이 나오는 걸 확인합니다.

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch10')

## 1. 산점도와 상관계수

웹 챕터와 같은 데이터로, 학생 8명의 하루 공부 시간과 시험 점수입니다.
상관계수 $r$ (−1 ~ +1)로 직선 관계의 방향과 강도를 숫자 하나로 요약합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'   # 한글 폰트 (Windows)
plt.rcParams['axes.unicode_minus'] = False

hours  = np.array([1, 2, 3, 4, 5, 6, 7, 8])          # 하루 공부 시간
scores = np.array([32, 44, 49, 55, 65, 62, 78, 83])  # 시험 점수 (웹 챕터와 동일)

r = np.corrcoef(hours, scores)[0, 1]
print(f'상관계수 r = {r:.3f}')   # ≈ 0.98 — 아주 강한 오르막

plt.figure(figsize=(6, 4))
plt.scatter(hours, scores, s=60, color='#4f46e5')
plt.title(f'공부 시간 vs 점수 (r = {r:.2f})')
plt.xlabel('하루 공부 시간 (시간)'); plt.ylabel('시험 점수')
plt.grid(alpha=0.3)
plt.show()

## 2. 최소제곱법 손계산: (1,1), (2,3), (3,5)

웹 챕터의 StepQuiz 와 같은 계산입니다:

$$a = \frac{\sum(x-\bar{x})(y-\bar{y})}{\sum(x-\bar{x})^2}, \qquad b = \bar{y} - a\,\bar{x}$$

In [ ]:
x = np.array([1, 2, 3])
y = np.array([1, 3, 5])

x_bar, y_bar = x.mean(), y.mean()
num = ((x - x_bar) * (y - y_bar)).sum()   # 분자: x 편차 × y 편차의 합
den = ((x - x_bar) ** 2).sum()            # 분모: x 편차 제곱의 합
a = num / den
b = y_bar - a * x_bar
print(f'x평균 = {x_bar}, y평균 = {y_bar}, 분자 = {num}, 분모 = {den}')
print(f'기울기 a = {a:.0f}, 절편 b = {b:.0f}  →  예측식 = {a:.0f}x {b:+.0f}')
print(f'x=4 의 예측값: {a * 4 + b:.0f}')

In [ ]:
# numpy 의 1차 다항식 피팅으로 검산 — 같은 답이 나와야 합니다
a2, b2 = np.polyfit(x, y, deg=1)
print(f'np.polyfit 검산: a = {a2:.4f}, b = {b2:.4f}  (손계산과 일치!)')

## 3. 진짜 데이터에 회귀선 긋기: 잔차 보기

공부 시간 데이터는 완벽한 직선 위에 있지 않으므로 **잔차(실제 − 예측)** 가 생깁니다.
최소제곱법은 잔차 제곱합(SSE)을 최소로 만드는 직선을 고른 것입니다.

In [ ]:
a3, b3 = np.polyfit(hours, scores, deg=1)
pred = a3 * hours + b3
resid = scores - pred
sse = (resid ** 2).sum()
print(f'회귀 직선: 예측 점수 = {a3:.2f} × 공부시간 + {b3:.2f}')
print(f'잔차: {np.round(resid, 1)}')
print(f'잔차 제곱합 SSE = {sse:.1f}')

plt.figure(figsize=(6, 4))
plt.scatter(hours, scores, s=60, color='#4f46e5', zorder=3, label='실제 데이터')
xs = np.linspace(0, 9, 50)
plt.plot(xs, a3 * xs + b3, color='#10b981', lw=2, label='최소제곱 직선')
plt.vlines(hours, pred, scores, color='#fb7185', ls='--', label='잔차')
plt.title('회귀 직선과 잔차'); plt.xlabel('하루 공부 시간 (시간)'); plt.ylabel('시험 점수')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 4. 경사하강법으로도 풀어보기 (2부 연결!)

회귀는 공식이 있지만, 신경망처럼 복잡한 모델은 공식이 없어서 **경사하강법**으로
손실 골짜기를 한 걸음씩 내려갑니다. 같은 문제를 경사하강법으로 풀어 봅시다.

먼저 **2장의 z점수 표준화**로 두 변수의 스케일을 맞춥니다. 스케일이 다르면
경사하강법이 잘 못 내려간다는 것, 2장 AI 연결에서 예고했던 바로 그 내용입니다.
(보너스: 표준화한 데이터의 회귀 기울기는 정확히 **상관계수 r** 이 됩니다!)

In [ ]:
# 표준화 (2장 z점수): 평균 0, 표준편차 1 로
zx = (hours - hours.mean()) / hours.std()
zy = (scores - scores.mean()) / scores.std()

a_gd, b_gd, lr = 0.0, 0.0, 0.1   # 아무 데서나 출발, 학습률 0.1
for step in range(1, 101):
    pred_z = a_gd * zx + b_gd
    err = pred_z - zy
    a_gd -= lr * 2 * (err * zx).mean()   # 손실을 a 로 미분한 값의 반대 방향 (2부!)
    b_gd -= lr * 2 * err.mean()          # 손실을 b 로 미분한 값의 반대 방향
    if step in (1, 5, 20, 100):
        print(f'{step:>3}걸음: 손실 = {(err ** 2).mean():.4f}, 기울기 a = {a_gd:.3f}')

print(f'\n경사하강법의 답 a ≈ {a_gd:.3f}  vs  상관계수 r = {np.corrcoef(hours, scores)[0, 1]:.3f}')
print('표준화한 데이터의 회귀 기울기 = 상관계수 r. 2장의 z점수가 여기서 다시 만납니다!')

## 5. 상관 ≠ 인과 (마지막 경고)

- 아이스크림 판매량과 익사 사고는 상관이 높지만, 원인은 둘 다 **더운 날씨**(숨은 변수).
- $r$ 이 높다는 것은 "같이 움직인다"까지만 말해 줄 뿐, **누가 원인인지는 알려주지 않습니다.**
- 인과를 확인하는 표준 도구는 9장의 **무작위 실험(A/B 테스트)** 입니다.

## 🎓 1~3부 완주!

방금 여러분은 하나의 문제(회귀)를 **통계의 공식**, **numpy**, 그리고 **경사하강법**으로
세 번 풀었습니다. 1~3부 전체가 한 문제 안에서 만난 것입니다.

| 시리즈 | 배운 것 | AI 에서의 역할 |
|--------|---------|----------------|
| 1부 선형대수 | 벡터·행렬 | 데이터와 가중치의 **구조** |
| 2부 미분 | 도함수·경사하강법·역전파 | **학습**의 원리 |
| 3부 확률과 통계 | 분포·표본·검정·회귀 | **데이터와 불확실성**의 언어 |

> **AI = 행렬로 표현된 거대한 함수를, 확률로 정의된 손실이 작아지도록,
> 미분으로 데이터에 맞춰가는 것.**

이 문장을 이제 전부 이해하실 수 있습니다. 완주를 축하합니다! 🎉